In [ ]:
import pandas as pd

df = pd.read_csv('oto_full_data.csv')
print(df.head())
print(df.info())

  Ngày đăng bài                                    Tên  Năm sản xuất  \
0    05/10/2025      Hyundai Kona 2.0 AT Đặc biệt 2019        2019.0   
1    15/10/2025      Mercedes-Benz E200 Exclusive 2022        2022.0   
2    24/09/2025  Hyundai Accent 1.4 MT Tiêu chuẩn 2022        2022.0   
3    14/10/2025     Mitsubishi Xpander AT Premium 2022        2022.0   
4    13/10/2025       Toyota Corolla Cross 1.8 V  2023        2023.0   

      Xuất xứ              Địa điểm Kiểu dáng Số km đã đi      Hộp số  \
0  Trong nước  Nam Từ Liêm - Hà Nội       SUV   80.000 km  Số tự động   
1  Trong nước                Tp.HCM     Sedan   26.000 km  Số tự động   
2  Trong nước      Tân An - Long An     Sedan  130.000 km      Số sàn   
3  Trong nước  Chí Linh - Hải Dương       SUV   83.000 km  Số tự động   
4  Trong nước  An Dương - Hải Phòng       SUV   43.000 km  Số tự động   

  Trạng thái Nhiên Liệu             Giá  
0      Xe cũ   Máy xăng       450 triệu  
1      Xe cũ   Máy xăng  1 tỉ 650 triệu  
2 

In [ ]:
# Làm sạch dữ liệu
def priceStrtoInt(series):
    def convert_one(priceStr):
        if(priceStr == 'nan'):
            return 0
        priceStr = str(priceStr)
        if("tỉ" not in priceStr and "triệu" not in priceStr):
            return 0
        price = 0
        priceStr = str(priceStr).replace(" tỉ","000").replace("triệu","").strip()
        nums = priceStr.split()
        price = sum(float(x) for  x in nums)
        return price
    return series.apply(convert_one)

def soKmdadiToInt(series):
    def convert_one(km):
        if km == '' or km == 'nan':
            return 0
        return float(str(km).replace('.','').replace(' km','').strip())
    return series.apply(convert_one)

In [ ]:
df_clean = df.copy()

# Áp dụng hàm làm sạch
df_clean['Giá'] = priceStrtoInt(df_clean['Giá'])
df_clean['Số km đã đi'] = soKmdadiToInt(df_clean['Số km đã đi'])

current_year = 2025
df_clean['Tuổi xe'] = current_year - df_clean['Năm sản xuất']
df_clean['Thương hiệu'] = df_clean['Tên'].str.split().str[0]
df_clean['Dòng xe'] = df_clean['Tên'].str.split().str[1]

# Loại bỏ các dòng mà giá không thể chuyển đổi được
df_clean = df_clean[df_clean['Giá'] > 0]

print("Dữ liệu sau khi làm sạch:")
print(df_clean[['Giá', 'Số km đã đi', 'Tuổi xe', 'Kiểu dáng', 'Thương hiệu', 'Dòng xe']].head())

Dữ liệu sau khi làm sạch:
      Giá  Số km đã đi  Tuổi xe Kiểu dáng    Thương hiệu  Dòng xe
0   450.0      80000.0      6.0       SUV        Hyundai     Kona
1  1650.0      26000.0      3.0     Sedan  Mercedes-Benz     E200
2   325.0     130000.0      3.0     Sedan        Hyundai   Accent
3   515.0      83000.0      3.0       SUV     Mitsubishi  Xpander
4   700.0      43000.0      2.0       SUV         Toyota  Corolla


In [ ]:
# Chuẩn bị dữ liệu
features = ['Tuổi xe', 'Số km đã đi', 'Kiểu dáng', 'Hộp số', 'Nhiên Liệu', 'Xuất xứ', 'Thương hiệu', 'Dòng xe']
target = 'Giá'

X_pre = df_clean[features]
y = df_clean[target]

# Chuyển đổi các cột phân loại thành số
categorical_cols = ['Kiểu dáng', 'Hộp số', 'Nhiên Liệu', 'Xuất xứ', 'Thương hiệu', 'Dòng xe']
counts_dongxe = X_pre['Dòng xe'].value_counts()
dongxe_phobien = counts_dongxe[counts_dongxe >= 30].index

X_pre['Dòng xe'] = X_pre['Dòng xe'].apply(lambda x: x if x in dongxe_phobien else 'Khác')
X = pd.get_dummies(X_pre, columns=categorical_cols, drop_first=True)

print("Dữ liệu đầu vào (X) sau khi One-Hot Encoding:")
print(X.head())

Dữ liệu đầu vào (X) sau khi One-Hot Encoding:
   Tuổi xe  Số km đã đi  Kiểu dáng_Convertible  Kiểu dáng_Coupe  \
0      6.0      80000.0                  False            False   
1      3.0      26000.0                  False            False   
2      3.0     130000.0                  False            False   
3      3.0      83000.0                  False            False   
4      2.0      43000.0                  False            False   

   Kiểu dáng_Crossover  Kiểu dáng_Hatchback  Kiểu dáng_MPV  Kiểu dáng_Minibus  \
0                False                False          False              False   
1                False                False          False              False   
2                False                False          False              False   
3                False                False          False              False   
4                False                False          False              False   

   Kiểu dáng_SUV  Kiểu dáng_Sedan  ...  Dòng xe_Matiz  Dòng xe_R

/tmp/ipython-input-484758279.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_pre['Dòng xe'] = X_pre['Dòng xe'].apply(lambda x: x if x in dongxe_phobien else 'Khác')


In [ ]:
from sklearn.model_selection import train_test_split

# Chia Dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Số lượng mẫu trong tập train: {len(X_train)}")
print(f"Số lượng mẫu trong tập test: {len(X_test)}")

Số lượng mẫu trong tập train: 1783
Số lượng mẫu trong tập test: 446


In [ ]:
import numpy as np

# Xử lý NaN
numeric_cols_in_X = X_train.select_dtypes(include=np.number).columns

# Điền NaN cho cả X_train và X_test
for col in numeric_cols_in_X:
    mean_value = X_train[col].mean()
    X_train[col] = X_train[col].fillna(mean_value)
    X_test[col] = X_test[col].fillna(mean_value)

print("Đã xử lý NaN")

Đã xử lý NaN


In [ ]:
from sklearn.preprocessing import StandardScaler

# Chuẩn hóa Dữ liệu
numeric_cols = ['Tuổi xe', 'Số km đã đi']

scaler = StandardScaler()

# FIT và TRANSFORM trên tập Train
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# Chỉ TRANSFORM trên tập Test
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Dữ liệu train sau khi scale:")
print(X_train.head())

Dữ liệu train sau khi scale:
       Tuổi xe  Số km đã đi  Kiểu dáng_Convertible  Kiểu dáng_Coupe  \
507  -0.187398    -0.921311                  False            False   
746  -0.788352    -0.688932                  False            False   
205   0.613874     0.214817                  False            False   
2018  1.214828    -0.818039                  False            False   
2106 -0.187398    -0.017576                  False            False   

      Kiểu dáng_Crossover  Kiểu dáng_Hatchback  Kiểu dáng_MPV  \
507                 False                False          False   
746                 False                False          False   
205                 False                False          False   
2018                False                False          False   
2106                False                 True          False   

      Kiểu dáng_Minibus  Kiểu dáng_SUV  Kiểu dáng_Sedan  ...  Dòng xe_Matiz  \
507               False           True            False  ...          Fals

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Xây dựng và huấn luyện model theo thuật toán K-nearest neighbors
model = KNeighborsRegressor(n_neighbors=7)
model.fit(X_train, y_train)

print("Model đã được huấn luyện thành công")

Model đã được huấn luyện thành công


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Đánh giá model
y_pred = model.predict(X_test)

# Đánh giá model
r2 = r2_score(y_test, y_pred)
print(f"Chỉ số R-squared (R2): {r2:.4f}")

mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae:.2f}")

mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error (MSE): {mse:.2f}")

results = pd.DataFrame({'Giá thực tế': y_test, 'Giá dự đoán': y_pred})
print("\nSo sánh giá thực tế và giá dự đoán (5 mẫu đầu tiên):")
print(results.head())

Chỉ số R-squared (R2): 0.6493
Mean Absolute Error (MAE): 233.94
Mean Squared Error (MSE): 625605.35

So sánh giá thực tế và giá dự đoán (5 mẫu đầu tiên):
      Giá thực tế  Giá dự đoán
56          550.0   701.142857
494        1050.0  1126.000000
1672        605.0   605.000000
218         545.0   486.857143
952         325.0   361.000000
